# TensorFlow の基礎

**学習目標**
  1. 定数 Tensor の定義と基本的な操作の実践
  1. TensorFlow の自動微分機能の活用
  1. TensorFlow を使用したスクラッチからの線形回帰トレーニング方法の学習

このノートブックでは、まず TensorFlow における Tensor の主な操作を確認し、TensorFlow の Variable の操作方法を理解します。また、これらが Python の組み込みリストや NumPy 配列とどのように互換性があるかを説明します。

次に、勾配降下法を用いてスクラッチから線形回帰をトレーニングする問題に取り組みます。まず最初に、関数の引数（ここではモデルの重み）に対する関数（ここでは損失）の勾配を計算する方法を理解します。これを可能にする TensorFlow の構造が `tf.GradientTape` であり、これについて解説します。

最後に、線形モデルから生成された合成データを使用して、1 次元の線形回帰の重みを学習するシンプルなトレーニング ループを作成します。

ボーナス演習として、非線形モデルから生成されたデータに対しても同様の処理を行い、線形モデルのパフォーマンスを向上させるために非線形特徴量を手動でエンジニアリングします。

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

In [ ]:
print(tf.__version__)

## Tensor に対する操作

### 変数（Variables）と定数（Constants）

TensorFlow の Tensor は、定数（`tf.constant`）または変数（`tf.Variable`）のいずれかです。

定数の値は変更できませんが、変数の値は変更できます。
主な違いは、`tf.Variable` のインスタンスには値を変更するためのメソッドが用意されているのに対し、`tf.constant` で構築された Tensor にはそれらのメソッドがなく、値の変更ができない点です。`tf.Variable` の値を変更したい場合は、以下のいずれかのメソッドを使用します。
* `x.assign(new_value)`
* `x.assign_add(value_to_be_added)`
* `x.assign_sub(value_to_be_subtracted)`

In [ ]:
x = tf.constant([2, 3, 4])
x

In [ ]:
x = tf.Variable(2.0, dtype=tf.float32, name="my_variable")

In [ ]:
x.assign(45.8)  # TODO 1
x

In [ ]:
x.assign_add(4)  # TODO 2
x

In [ ]:
x.assign_sub(3)  # TODO 3
x

### 要素ごとの操作（Point-wise operations）

TensorFlow は、NumPy と同様の要素ごとの（point-wise）Tensor 操作を提供しています。  
* `tf.add`: Tensor の要素同士の加算
* `tf.multiply`: Tensor の要素同士の乗算
* `tf.subtract`: Tensor の要素同士の減算
* `tf.math.*`: Tensor の要素に適用される一般的な数学演算
* その他多数...

ほとんどの標準的な算術演算（`tf.add`、`tf.subtract` など）は、対応する通常の算術記号（`+`、`-` など）でオーバーロードされています。

In [ ]:
a = tf.constant([5, 3, 8])  # TODO 1
b = tf.constant([3, -1, 2])
c = tf.add(a, b)
d = a + b

print("c:", c)
print("d:", d)

In [ ]:
a = tf.constant([5, 3, 8])  # TODO 2
b = tf.constant([3, -1, 2])
c = tf.multiply(a, b)
d = a * b

print("c:", c)
print("d:", d)

In [ ]:
# tf.math.exp は浮動小数点数を想定しているため、明示的に型を指定する必要があります
a = tf.constant([5, 3, 8], dtype=tf.float32)
b = tf.math.exp(a)

print("b:", b)

### NumPy との相互運用性

ネイティブの TF Tensor に加えて、TensorFlow の操作はネイティブの Python 型や NumPy 配列をオペランドとして受け取ることができます。

In [ ]:
# ネイティブの Python リスト
a_py = [1, 2]
b_py = [3, 4]

In [ ]:
tf.add(a_py, b_py)  # TODO 1

In [ ]:
# NumPy 配列
a_np = np.array([1, 2])
b_np = np.array([3, 4])

In [ ]:
tf.add(a_np, b_np)  # TODO 2

In [ ]:
# ネイティブの TF Tensor
a_tf = tf.constant([1, 2])
b_tf = tf.constant([3, 4])

In [ ]:
tf.add(a_tf, b_tf)  # TODO 3

`.numpy()` を使用して、ネイティブの TF Tensor を NumPy 配列に変換できます。

In [ ]:
a_tf.numpy()

## 線形回帰

ここでは、低レベルの TensorFlow 操作を使用して線形回帰を実装してみましょう。

コースの後半では、高レベルの TensorFlow を使用したより抽象化された方法を学習します。

### トイ データセット

次の関数をモデル化します。

\begin{equation}
y= 2x + 10
\end{equation}

In [ ]:
X = tf.constant(range(10), dtype=tf.float32)
Y = 2 * X + 10

print(f"X:{X}")
print(f"Y:{Y}")

モデルを評価するためのテスト データセットも作成しましょう。

In [ ]:
X_test = tf.constant(range(10, 20), dtype=tf.float32)
Y_test = 2 * X_test + 10

print(f"X_test:{X_test}")
print(f"Y_test:{Y_test}")

#### 損失関数

構築できる最も単純なモデルは、x の各値に対してトレーニング セットの標本平均を返すモデルです。

In [ ]:
# トレーニング データの標本平均を計算
y_mean = Y_test.numpy().mean()


def predict_mean(X):
    y_hat = [y_mean] * len(X)
    return y_hat


# テスト データの y_hat を計算
Y_hat = predict_mean(X_test)

平均二乗誤差（MSE）を使用すると、損失は次のようになります。
\begin{equation}
MSE = \frac{1}{m}\sum_{i=1}^{m}(\hat{Y}_i-Y_i)^2
\end{equation}

この単純なモデルの場合、損失は次のようになります。

In [ ]:
errors = (Y_hat - Y_test) ** 2
loss = tf.reduce_mean(errors)
loss.numpy()

上記の MSE 損失の値は、より複雑なモデルの性能を比較するためのベースラインになります。

ここで、線形回帰モデルを使用したときのモデルの予測値を含むベクトルを $\hat{Y}$ と表すと、
\begin{equation}
\hat{Y} = w_0X + w_1
\end{equation}

モデルの係数を引数として取る損失関数を次のように記述できます。

In [ ]:
def loss_mse(X, Y, w0, w1):
    Y_hat = w0 * X + w1
    errors = (Y_hat - Y) ** 2
    return tf.reduce_mean(errors)

### 勾配関数

勾配降下法を使用するには、各重みに対する損失関数の偏微分を求める必要があります。手動で微分を計算することもできますが、TensorFlow の自動微分機能があるため、その必要はありません。

勾配降下法では、損失をパラメータ $w_0$ と $w_1$ の関数として考えます。したがって、これらの変数に関する偏微分を計算したいと考えます。

そのためには、勾配情報を記録する `tf.GradientTape` インスタンスのコンテキスト内で損失の計算をラップする必要があります。

```python
with tf.GradientTape() as tape:
    loss = # computation 
```

これにより、後で `tf.GradientTape` コンテキスト内で計算された任意の Tensor について、`tf.Variable` のインスタンスに対する勾配を計算できるようになります。

```python
gradients = tape.gradient(loss, [w0, w1])
```

モデルの重みに対する損失の勾配を計算することで、この手順を示します。

In [ ]:
# TODO 1
def compute_gradients(X, Y, w0, w1):
    with tf.GradientTape() as tape:
        loss = loss_mse(X, Y, w0, w1)
    return tape.gradient(loss, [w0, w1])

In [ ]:
w0 = tf.Variable(0.0)
w1 = tf.Variable(0.0)

dw0, dw1 = compute_gradients(X, Y, w0, w1)

In [ ]:
print("dw0:", dw0.numpy())

In [ ]:
print("dw1", dw1.numpy())

### トレーニング ループ

ここでは、収束する非常にシンプルなトレーニング ループを作成します。なお、単純化のため、バッチ処理、個別のテスト セットの作成、重みのランダム初期化などのベスト プラクティスは省略しています。

In [ ]:
STEPS = 1000
LEARNING_RATE = 0.02
MSG = "STEP {step} - loss: {loss}, w0: {w0}, w1: {w1}\n"


w0 = tf.Variable(0.0)
w1 = tf.Variable(0.0)


for step in range(1, STEPS + 1):
    dw0, dw1 = compute_gradients(X, Y, w0, w1)

    w0.assign_sub(dw0 * LEARNING_RATE)
    w1.assign_sub(dw1 * LEARNING_RATE)

    if step % 100 == 0:
        loss = loss_mse(X, Y, w0, w1)
        print(MSG.format(step=step, loss=loss, w0=w0.numpy(), w1=w1.numpy()))

それでは、この線形回帰のテスト損失と、常にトレーニング セットの平均値を出力するベースライン モデルのテスト損失を比較してみましょう。

In [ ]:
loss = loss_mse(X_test, Y_test, w0, w1)
loss.numpy()

確かに大幅に改善されています。

## ボーナス

次のような非線形関数のモデル化を試してみましょう: $y=xe^{-x^2}$

In [ ]:
X = tf.constant(np.linspace(0, 2, 1000), dtype=tf.float32)
Y = X * tf.exp(-(X**2))

In [ ]:
%matplotlib inline

plt.plot(X, Y)

In [ ]:
def make_features(X):
    f1 = tf.ones_like(X)  # バイアス
    f2 = X
    f3 = tf.square(X)
    f4 = tf.sqrt(X)
    f5 = tf.exp(X)
    return tf.stack([f1, f2, f3, f4, f5], axis=1)

In [ ]:
def predict(X, W):
    return tf.squeeze(X @ W, -1)

In [ ]:
def loss_mse(X, Y, W):
    Y_hat = predict(X, W)
    errors = (Y_hat - Y) ** 2
    return tf.reduce_mean(errors)

In [ ]:
def compute_gradients(X, Y, W):
    with tf.GradientTape() as tape:
        loss = loss_mse(X, Y, W)
    return tape.gradient(loss, W)

In [ ]:
STEPS = 2000
LEARNING_RATE = 0.02


Xf = make_features(X)
n_weights = Xf.shape[1]

W = tf.Variable(np.zeros((n_weights, 1)), dtype=tf.float32)

# プロット用
steps, losses = [], []
plt.figure()


for step in range(1, STEPS + 1):
    dW = compute_gradients(Xf, Y, W)
    W.assign_sub(dW * LEARNING_RATE)

    if step % 100 == 0:
        loss = loss_mse(Xf, Y, W)
        steps.append(step)
        losses.append(loss)
        plt.clf()
        plt.plot(steps, losses)


print(f"STEP: {STEPS} MSE: {loss_mse(Xf, Y, W)}")

plt.figure()
plt.plot(X, Y, label="actual")
plt.plot(X, predict(Xf, W), label="predicted")
plt.legend()

Copyright 2021 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.